# Notebook 10-Experiment 18: Imbalance Correction Comparison
### Novelty 1
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

TweetEval's negative class is only about 19% of the data, so a model trained
straight on it under-predicts negatives. I wanted to compare the two usual fixes
rather than just pick one.

Model-level fix: Complement Naive Bayes, which learns from the complement of each
class so its estimates hold up when the classes are skewed. Data-level fix: SMOTE,
which makes synthetic minority examples until things balance out.

Most of the papers I read used one or the other and reported overall accuracy. The
thing I actually care about here is whether either fix helps the sarcasm and slang
subgroups — a correction that lifts overall accuracy but leaves sarcasm just as bad
hasn't fixed the problem this thesis is about. So I run both and compare them on
subgroup fairness, not accuracy.

Three quick Naive Bayes training runs. Fills **Table 7**.

## Cell 1: Setup

In [ ]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.4 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal


## Cell 2: Load Data and the Exp 3 Baseline

Experiment 3 already trained a plain Multinomial NB. That result is the reference
point every condition here is measured against.

In [ ]:
D = PATHS["data"]

tw_train = pd.read_parquet(D / "tw_train.parquet")
tw_test  = pd.read_parquet(D / "tw_test.parquet")

X_train = sp.load_npz(D / "tw_Xtrain_fs1.npz")
X_test  = sp.load_npz(D / "tw_Xtest_fs1.npz")

y_train = tw_train["sentiment"].values
y_test  = tw_test["sentiment"].values
sg_test = tw_test["subgroup_primary"].values

print("Class distribution in training data:")
dist = pd.Series(y_train).value_counts()
for cls, n in dist.items():
    print(f"  {cls:<10}: {n:>7,}  ({n/len(y_train)*100:>5.1f}%)")
print(f"\nImbalance ratio (majority : minority) = "
      f"{dist.max()/dist.min():.2f} : 1")

baseline_pred = load_predictions("exp03", "MultinomialNB")
print(f"\nExp 3 baseline loaded: {len(baseline_pred):,} predictions")

Class distribution in training data:
  neutral   :  20,673  ( 45.3%)
  positive  :  17,849  ( 39.1%)
  negative  :   7,093  ( 15.5%)

Imbalance ratio (majority : minority) = 2.91 : 1

Exp 3 baseline loaded: 12,284 predictions


## Cell 3: Condition Runner

SMOTE is applied to the training partition only, after vectorisation. Applying it
before the split, or to the test set, would leak synthetic examples into
evaluation and invalidate every result downstream.

In [ ]:
!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE
from sklearn.naive_bayes import MultinomialNB, ComplementNB

def run_condition(exp_id, label, model, use_smote=False):
    print("="*62)
    print(label)
    print("="*62)

    Xtr, ytr = X_train, y_train

    if use_smote:
        # k_neighbors kept low because the minority class is sparse text
        sm = SMOTE(random_state=SEED, k_neighbors=3)
        Xtr, ytr = sm.fit_resample(X_train, y_train)
        print(f"  SMOTE applied: {X_train.shape[0]:,} -> {Xtr.shape[0]:,} rows")
        after = pd.Series(ytr).value_counts()
        print(f"  balanced to: {dict(after)}")

    t0 = time.time()
    model.fit(Xtr, ytr)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    elapsed = time.time() - t0

    res = evaluate_model(y_test, y_pred, y_proba, label=label)
    res["Condition"]     = label
    res["Dataset"]       = "TweetEval"
    res["Fit Time (s)"]  = round(elapsed, 1)

    # negative class recall is the direct measure of whether the
    # correction actually helped the minority class
    from sklearn.metrics import recall_score
    res["Negative Class Recall"] = recall_score(
        y_test, y_pred, labels=["negative"], average="macro", zero_division=0)

    save_model(model, exp_id, label.replace(" ", "_"))
    save_predictions(exp_id, label.replace(" ", "_"),
                     y_test, y_pred, y_proba, sg_test)

    print(f"\n  Macro F1              : {res['Macro F1']:.4f}")
    print(f"  Negative class recall : {res['Negative Class Recall']:.4f}")
    print(f"  Fit time              : {elapsed:.1f}s\n")
    return res

conditions = []

## Cell 4: Four Conditions

The baseline row is reconstructed from the saved Experiment 3 predictions rather
than retrained, so it is exactly the same model that appears in Table 1.

In [ ]:
from sklearn.metrics import recall_score

# Condition A — reference, from saved Exp 3 output
base_res = evaluate_model(baseline_pred["y_true"].values,
                          baseline_pred["y_pred"].values,
                          label="Multinomial NB (Exp 3 baseline)")
base_res["Condition"] = "Multinomial NB (Exp 3 baseline)"
base_res["Dataset"]   = "TweetEval"
base_res["Negative Class Recall"] = recall_score(
    baseline_pred["y_true"], baseline_pred["y_pred"],
    labels=["negative"], average="macro", zero_division=0)
conditions.append(base_res)
print(f"Condition A — baseline macro F1: {base_res['Macro F1']:.4f}\n")

# Condition B — data-level correction
conditions.append(run_condition("exp18b", "Multinomial NB + SMOTE",
                                MultinomialNB(alpha=0.5), use_smote=True))

# Condition C — model-level correction
conditions.append(run_condition("exp18c", "Complement NB",
                                ComplementNB(alpha=0.5), use_smote=False))

# Condition D — both
conditions.append(run_condition("exp18d", "Complement NB + SMOTE",
                                ComplementNB(alpha=0.5), use_smote=True))

Condition A — baseline macro F1: 0.5589

Multinomial NB + SMOTE
  SMOTE applied: 45,615 -> 62,019 rows
  balanced to: {'positive': np.int64(20673), 'neutral': np.int64(20673), 'negative': np.int64(20673)}
  saved model -> exp18b_Multinomial_NB_+_SMOTE.pkl
  saved predictions -> exp18b_Multinomial_NB_+_SMOTE_TweetEval.parquet  (12,284 rows)

  Macro F1              : 0.5498
  Negative class recall : 0.7550
  Fit time              : 0.4s

Complement NB
  saved model -> exp18c_Complement_NB.pkl
  saved predictions -> exp18c_Complement_NB_TweetEval.parquet  (12,284 rows)

  Macro F1              : 0.5677
  Negative class recall : 0.6619
  Fit time              : 0.2s

Complement NB + SMOTE
  SMOTE applied: 45,615 -> 62,019 rows
  balanced to: {'positive': np.int64(20673), 'neutral': np.int64(20673), 'negative': np.int64(20673)}
  saved model -> exp18d_Complement_NB_+_SMOTE.pkl
  saved predictions -> exp18d_Complement_NB_+_SMOTE_TweetEval.parquet  (12,284 rows)

  Macro F1              : 0.

## Cell 5: Fairness Comparison — the Actual Point

Accuracy is not the criterion here. What matters is the fairness gap between each
informal subgroup and the formal reference group, and whether either correction
strategy narrows it.

In [ ]:
!pip install -q aif360 fairlearn

import importlib, sys
for m in list(sys.modules):
    if m.startswith(("aif360", "fairlearn")):
        del sys.modules[m]

try:
    from aif360.datasets import BinaryLabelDataset
    print("AIF360 OK")
except Exception as e:
    print(f"AIF360 failed: {e}")

try:
    from fairlearn.metrics import demographic_parity_difference
    print("Fairlearn OK")
except Exception as e:
    print(f"Fairlearn failed: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.9 MB/s eta 0:00:00
AIF360 OK
Fairlearn OK


In [ ]:
ids = [("exp03",  "MultinomialNB",           "Multinomial NB (Exp 3 baseline)"),
       ("exp18b", "Multinomial_NB_+_SMOTE",  "Multinomial NB + SMOTE"),
       ("exp18c", "Complement_NB",           "Complement NB"),
       ("exp18d", "Complement_NB_+_SMOTE",   "Complement NB + SMOTE")]

rows = []
for exp_id, fname, label in ids:
    try:
        pdf = load_predictions(exp_id, fname)
    except FileNotFoundError:
        print(f"  missing: {exp_id} {fname}"); continue

    audit = full_fairness_audit(pdf)
    rep   = subgroup_report(pdf)

    row = {"Condition": label}
    if not audit.empty and "AIF360 AOD" in audit.columns:
        for _, a in audit.iterrows():
            row[f"{a['Subgroup']} AOD"] = a.get("AIF360 AOD", np.nan)
        row["Mean AOD"] = audit["AIF360 AOD"].abs().mean()
        sar = audit[audit["Subgroup"] == "sarcasm"]
        row["Sarcasm DIR"] = sar.iloc[0].get("AIF360 DIR", np.nan) if not sar.empty else np.nan

    for _, r in rep.iterrows():
        row[f"{r['Subgroup']} F1"] = r["Macro F1"]

    rows.append(row)

fair_df = pd.DataFrame(rows)
print("SUBGROUP FAIRNESS BY CORRECTION STRATEGY")
print("="*80)
print(fair_df.round(4).to_string(index=False))

pip install 'aif360[inFairness]'


SUBGROUP FAIRNESS BY CORRECTION STRATEGY
                      Condition  emoji-heavy AOD  slang-heavy AOD  sarcasm AOD  Mean AOD  Sarcasm DIR  formal F1  other F1  emoji-heavy F1  slang-heavy F1  sarcasm F1
Multinomial NB (Exp 3 baseline)          -0.0141           0.0023      -0.0018    0.0061       0.9941     0.5517    0.5464          0.5615          0.5765      0.5460
         Multinomial NB + SMOTE          -0.0142           0.0094      -0.0223    0.0153       0.7685     0.5503    0.4791          0.5257          0.5096      0.3903
                  Complement NB          -0.0111           0.0194      -0.0360    0.0222       0.6302     0.5637    0.5290          0.5653          0.6318      0.3444
          Complement NB + SMOTE          -0.0125           0.0005      -0.0196    0.0109       0.7858     0.5379    0.4826          0.5179          0.4926      0.3979


## Cell 6: Table 7

In [ ]:
table7 = pd.DataFrame(conditions)[
    ["Condition", "Dataset", "Macro F1", "Accuracy", "Negative Class Recall"]
].merge(fair_df, on="Condition", how="left")

if "Mean AOD" in table7.columns:
    baseline_aod = table7.iloc[0]["Mean AOD"]
    table7["Better Than Baseline?"] = [
        "reference" if i == 0
        else ("Yes" if row["Mean AOD"] < baseline_aod else "No")
        for i, row in table7.iterrows()
    ]

    best = table7.iloc[1:]["Mean AOD"].idxmin()
    print("="*90)
    print("TABLE 7 — IMBALANCE CORRECTION COMPARISON")
    print("="*90)
    print(table7.round(4).to_string(index=False))
    print(f"\nLowest mean AOD: {table7.loc[best, 'Condition']}")
    print(f"  mean AOD  : {table7.loc[best, 'Mean AOD']:.4f} "
          f"(baseline {baseline_aod:.4f})")
    print(f"  macro F1  : {table7.loc[best, 'Macro F1']:.4f} "
          f"(baseline {table7.iloc[0]['Macro F1']:.4f})")
    print("\nIf the fairest condition is not the most accurate, that tension")
    print("is the finding — it is the accuracy cost of fairer treatment.")

save_result_table(table7, "Table7_Imbalance_Correction")
print("\nNovelty 1 complete.")

TABLE 7 — IMBALANCE CORRECTION COMPARISON
                      Condition   Dataset  Macro F1  Accuracy  Negative Class Recall  emoji-heavy AOD  slang-heavy AOD  sarcasm AOD  Mean AOD  Sarcasm DIR  formal F1  other F1  emoji-heavy F1  slang-heavy F1  sarcasm F1 Better Than Baseline?
Multinomial NB (Exp 3 baseline) TweetEval    0.5589    0.5791                 0.4021          -0.0141           0.0023      -0.0018    0.0061       0.9941     0.5517    0.5464          0.5615          0.5765      0.5460             reference
         Multinomial NB + SMOTE TweetEval    0.5498    0.5500                 0.7550          -0.0142           0.0094      -0.0223    0.0153       0.7685     0.5503    0.4791          0.5257          0.5096      0.3903                    No
                  Complement NB TweetEval    0.5677    0.5655                 0.6619          -0.0111           0.0194      -0.0360    0.0222       0.6302     0.5637    0.5290          0.5653          0.6318      0.3444             